# Statcast data fetch

Pulls MLB regular-season Statcast data to `data/<year>_data.csv`, one file
per season, for the swing-decision model.

Each season is filtered on two things:

1. `game_type == 'R'` — regular season only.
2. **Games outside the US and Canada are dropped.** International series are
   played at neutral sites where the tracking system is a temporary
   installation, so calibration may differ from a regular park. This removes
   Mexico City (2023, 2024, 2026), London (2023, 2024), Seoul (2024) and Tokyo
   (2025) — 14 games across 2021–2026. Toronto is **kept**: Rogers Centre is a
   permanent park with a permanent Hawk-Eye rig.

The pitch-level Statcast export has no venue column, and international series
keep an MLB club as `home_team`, so neither column identifies these games. The
filter resolves `game_pk` to a venue through the MLB Stats API instead.

Season date ranges also come from the API rather than being hardcoded, so the
window is the actual regular season for each year.

In [ ]:
import sys; sys.path.insert(0, '..')

from src.data import fetch_season, season_bounds, game_venues, KEEP_COUNTRIES, DATA_DIR

print(f'writing to {DATA_DIR}')

The fetch logic lives in `src/data.py` rather than here, so the season bounds,
the venue lookup and the exclusion rule are defined once and the cleaning code
that reads these files shares them.

In [ ]:
# Edit YEARS to pull only what you need -- each season is a slow download.
YEARS = range(2021, 2027)

for year in YEARS:
    fetch_season(year)

### Which games are excluded

Regenerated from the Stats API rather than hardcoded, so it stays correct as
seasons are added.

In [ ]:
import pandas as pd

venues = pd.concat([game_venues(y).assign(season=y) for y in YEARS])
overseas = venues[~venues.country.isin(KEEP_COUNTRIES)]
print(f'{len(overseas)} games excluded of {len(venues):,}')
overseas.groupby(['season', 'country', 'venue']).size().rename('games').to_frame()

## Notes

- **2026 is incomplete** until the regular-season finale (2026-09-27). Running
  the cell above before then pulls through today and says so; re-run afterwards
  to top it up.
- Location features are **not** comparable across the 2025/2026 boundary
  without harmonization: `plate_x`/`plate_z` moved from front-of-plate to
  middle-of-plate in 2026, and `sz_top`/`sz_bot` switched to the ABS-defined
  zone. The conversion (`src/data.to_middle_of_plate`) belongs in the cleaning
  step, not here; this notebook writes raw pulls. See `FINDINGS.md`.
- `data/` is gitignored, so these files stay local.